# Belief-persistence evaluation on Colab

**Before running:**
1. *Runtime → Change runtime type → T4 GPU*.
2. Zip the extracted images and upload the zip to Google Drive at `MyDrive/cmpe249/`.
   From the repo root, with the positive and control images extracted under `data/nuscenes/samples/CAM_FRONT/` (see the README):
   ```bash
   cd data/nuscenes && zip -qr ../nuscenes_samples.zip samples
   ```
   Upload the resulting `data/nuscenes_samples.zip` (1,254 images, about 185 MB).
   Its paths must start with `samples/CAM_FRONT/`, which zipping `samples` from inside `data/nuscenes` gives.

Results are written to that Drive folder, and `run` skips finished (sequence, mode) pairs, so if Colab disconnects or the free-tier GPU quota runs out, reconnect later and run the cells again; it continues where it stopped.
Every result line records the run settings and package versions, and `run` refuses to resume a file written with different ones.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Settings

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/cmpe249"   # holds nuscenes_samples.zip; results go here too
BRANCH = "feat/planted-belief"

# Run once per model: Qwen first, then switch to the LLaVA line and run from step 2 again.
# MAX_SIDE = longest image side fed to the model. LLaVA-OneVision tiles each image, so a 6-frame
# history at 1024 px is ~27.7k tokens and runs out of memory on a T4; 768 px is ~14.9k.
# (Qwen at 1024 px is ~5.3k.) Keep MAX_SIDE fixed for a model once its run has started.
MODEL, TAG, MAX_SIDE = "Qwen/Qwen2.5-VL-7B-Instruct", "qwen7b", 1024
# MODEL, TAG, MAX_SIDE = "llava-hf/llava-onevision-qwen2-7b-ov-hf", "llava7b", 768

RESULTS = f"{DRIVE_DIR}/results_{TAG}_v2.jsonl"   # v2: rerun with the "most recent image" prompt
RUN = (f"PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m belief_eval run --dataroot data/nuscenes "
       f"--model {MODEL} --4bit --max-side {MAX_SIDE}")

def progress():
    """Finished records per (sequence kind, mode) in the results file."""
    import collections, json, os
    if os.path.exists(RESULTS):
        counts = collections.Counter((r.get("kind", "positive"), r["mode"]) for r in map(json.loads, open(RESULTS)))
        for (kind, mode), n in sorted(counts.items()):
            print(f"{kind:9} {mode:12} {n}")

## 2. Setup: Drive, code, packages, images

Package versions are pinned in `requirements.txt` (torch stays at Colab's preinstalled build).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!if [ -d /content/repo ]; then git -C /content/repo fetch -q && git -C /content/repo checkout -q {BRANCH} && git -C /content/repo pull -q; else git clone -q -b {BRANCH} https://github.com/n-zad/av-vlm-belief-persistence.git /content/repo; fi
%cd /content/repo
!pip install -q -r requirements.txt
!pip list 2>/dev/null | grep -E "^(torch|transformers|accelerate|bitsandbytes) "
!mkdir -p data/nuscenes && unzip -q -o {DRIVE_DIR}/nuscenes_samples.zip -d data/nuscenes

# every image the sequences use must be at data/nuscenes/samples/CAM_FRONT/
import json, os
for name in ("sequences_trainval.jsonl", "sequences_control.jsonl"):
    imgs = {fr["image"] for s in map(json.loads, open(name)) for fr in s["frames"]}
    found = sum(os.path.exists(os.path.join("data/nuscenes", i)) for i in imgs)
    print(f"{name}: {found} / {len(imgs)} images present" + ("" if found == len(imgs) else "  <-- missing images"))

## 3. Smoke test: one sequence, all five modes

Check that the raw answers are sensible JSON before the long run.
If they are empty or gibberish, stop here and report it.
If you get a CUDA out-of-memory error, lower `MAX_SIDE` in step 1 (e.g. 640).
Planted frames (`planted=True`) are fixed answers written into the history, not model output.

In [ ]:
!rm -f /content/smoke.jsonl
!{RUN} --sequences sequences_trainval.jsonl --limit 1 --out /content/smoke.jsonl

import json
for rec in map(json.loads, open("/content/smoke.jsonl")):
    print(f"--- {rec['mode']} (evidence frame {rec['evidence_index']})")
    for f in rec["frames"]:
        print(f"t={f['t']} present={f['present']} action={f['action']} parsed_from={f['parsed_from']} "
              f"planted={f['planted']} | {f['raw'][:120]!r}")
print(rec["config"])

## 4. Full run, in priority order

Each step is safe to re-run after a disconnect.
Run them in order: if the GPU quota runs out, what is left unfinished is the least important part.
If you change `MAX_SIDE` after a partial run, delete that model's results file in Drive first (the run refuses to mix settings).

**4a. Core results** (stateless, stateful, planted "no"): 145 sequences × 3 modes.
Each positive mode should reach 145.

In [ ]:
!{RUN} --sequences sequences_trainval.jsonl --modes stateless stateful planted_no --out {RESULTS}
progress()

**4b. Planted "yes" and the reconsider mitigation.**

In [ ]:
!{RUN} --sequences sequences_trainval.jsonl --modes planted_yes reconsider --out {RESULTS}
progress()

**4c. Control sequences without history** (pedestrian-free frames: the false-positive rate).

In [ ]:
!{RUN} --sequences sequences_control.jsonl --modes stateless --out {RESULTS}
progress()

**4d. Planted "yes" on control sequences** (does a wrong "yes" persist on empty frames?).

In [ ]:
!{RUN} --sequences sequences_control.jsonl --modes planted_yes --out {RESULTS}
progress()

## 5. Scores (all models run so far)

Includes median latency, longest prompt and peak GPU memory per mode.

In [ ]:
!python -m belief_eval score {DRIVE_DIR}/results_*_v2.jsonl